## Reasoning Patch

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

### Set-up

In [2]:
import torch
import gc
import random
import pandas as pd
from tqdm import tqdm

import sys
sys.path.append("src")
import _util
import _dataset
import _prompt
import _mapping
from _intervention import prepare_batch_multitoken_intervention, batch_intervene, get_attention_freeze_hooks, prepare_batch_multitoken_head_intervention

## Experiment Config

In [3]:
model_type = "GPT-OSS_stepwise" # GPT-OSS or R1
freeze_attention = False
# Set if freeze attention is True————————
num_attention = 20
dataset_fn = _dataset.create_h_dataset
num_digits = 3
prompt_fn = _prompt.get_stepwise_prompt
divide_num = 22
modifier_fn = lambda x, y: x
# ———————————————————————————————————————
prompt_type = "h_pre_penultimate_sum" # {null / h / h1 / h2}_{null / pre_result / pre_final_sum / ...}
intervention_loc = "" # restatement or reasoning or restatement_and_reasoning
intervention_ids = [20]
tok_pos_fn = _mapping.intervene_id_to_tok_pos_stepwise_3_digit_h
result_dir = "layerwise"

## Set up Experiment

In [4]:
if "GPT-OSS" in model_type:
    model, tokenizer = _util.load_OSS()
elif "R1" in model_type:
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
def intervene_on_final_sum(prompt, add_ds_entry):
    print(prompt)
    source_prompt = prompt_fn(add_ds_entry["source_1_digits"], add_ds_entry["source_2_digits"], add_ds_entry["source_1_num"], add_ds_entry["source_2_num"])
    intervened_prompt = _prompt.get_intervened_prompt(intervention_ids, prompt, source_prompt)
    print(intervened_prompt)
    
    return intervened_prompt

modifier_fn = intervene_on_final_sum

In [5]:
attention_freeze_hooks = []
if freeze_attention:
    random.seed(42)
    add_ds = dataset_fn(num_digits=num_digits, num_samples=276)

    attention_prompts = []
    for add_ds_entry in add_ds[-20:]:
        base_prompt = prompt_fn(add_ds_entry["base_1_digits"], add_ds_entry["base_2_digits"], add_ds_entry["base_1_num"], add_ds_entry["base_2_num"])
        truncated_base_prompt = _prompt.divide_prompt(divide_num, base_prompt)[0]
        truncated_base_prompt = modifier_fn(truncated_base_prompt, add_ds_entry)
        print(truncated_base_prompt)
        
        attention_prompts.append(truncated_base_prompt)

    attention_tokens = tokenizer(attention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    attention_freeze_hooks = get_attention_freeze_hooks(model, attention_tokens)

In [6]:
if prompt_type:
    prompt_type = "_" + prompt_type
if len(intervention_loc) > 0:
    intervention_loc = "_" + intervention_loc

# Load the divided prompts dataset
if 'h1' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h1_prompts{prompt_type[3:]}.csv")
elif 'h2' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h2_prompts{prompt_type[3:]}.csv")
elif 'h' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h_prompts{prompt_type[2:]}.csv")
else:
    prompts = pd.read_csv(f"data/{model_type}/prompts{prompt_type}.csv")
    
prompts["base_sum"] = prompts["base_sum"].astype('Int64')
prompts["source_sum"] = prompts["source_sum"].astype('Int64')
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [7]:
if intervention_ids == None:
    if 'h' in prompt_type:
        if model_type == "GPT-OSS_stepwise":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit_h
        elif model_type == "GPT-OSS_vanilla":
            intervention_ids_dict = _mapping.intervene_ids_vanilla_2_digit_h
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit_h
        else:
            raise ValueError(f"Invalid model type: {model_type}")
    else:
        if model_type == "GPT-OSS_stepwise":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit
        else:
            raise ValueError(f"Invalid model type: {model_type}")

    if intervention_loc == "restatement":
        intervention_ids = intervention_ids_dict["restatement"]
    elif intervention_loc == "reasoning":
        intervention_ids = intervention_ids_dict["reasoning"]
    elif intervention_loc == "restatement_and_reasoning":
        intervention_ids = intervention_ids_dict["restatement"] + intervention_ids_dict["reasoning"]
    else:
        raise ValueError(f"Invalid intervention location: {intervention_loc}")

print(intervention_ids)

[20]


In [9]:
# for i, row in prompts.iterrows():
#     print(list(enumerate(tokenizer.convert_ids_to_tokens(tokenizer(row["base_prompt"], add_special_tokens=False, return_tensors="pt")["input_ids"][0]))))

In [8]:
tok_pos_list = [tok_pos_fn[id] for id in intervention_ids]
tok_pos_list = [tok_pos_fn[divide_num]-1]
print(tok_pos_list)

[235]


## Run Experiment

In [9]:
if freeze_attention:

    header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'layer', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
    filepath = _util.create_csv_file(f"experiments/activation_intervention/output/{model_type}/{result_dir}", f"{prompt_type[1:]}{intervention_loc}.csv", header, overwrite=False)

    # Batch size 1, because we run 20 attention patterns at once
    for i, row in tqdm(prompts.iterrows(), total=len(prompts)):
        # Preparing prompts and labels
        base_prompt = row['base_prompt']
        base_prompts = [row['base_prompt']] * len(attention_prompts)
        if "factual_output" in row and pd.notna(row["factual_output"]):
            factual_labels_str = [str(row['factual_output'])] * len(attention_prompts)
        else:
            factual_labels_str = [str(row['base_sum'])] * len(attention_prompts)
        factual_labels = tokenizer(factual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)
        source_prompt = row['source_prompt']
        source_prompts = [row['source_prompt']] * len(attention_prompts)
        if "counterfactual_output" in row and pd.notna(row["counterfactual_output"]):
            counterfactual_labels_str = [str(row['counterfactual_output'])] * len(attention_prompts)
        else:
            counterfactual_labels_str = [str(row['source_sum'])] * len(attention_prompts)
        counterfactual_labels = tokenizer(counterfactual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)

        # Preparing intervention hooks
        for layer in range(len(model.model.layers)):
            for head in range(64):
                intervene_hooks = []
                tokens, source_tokens, hook = prepare_batch_multitoken_intervention(model, tokenizer, layer, tok_pos_list, base_prompts, source_prompts, pre_hook=False, module_format=f"model.layers.{layer}.self_attn")
                intervene_hooks.append(hook)
                input_length = tokens["input_ids"].shape[1]

                # Forward pass
                with torch.no_grad():
                    output = batch_intervene(model, tokens["input_ids"], attention_freeze_hooks + intervene_hooks, attention_mask=tokens["attention_mask"], pre_hook=False)
                pred_toks = output.logits[:,-1,:].argmax(dim=-1)
                prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
                factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
                counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
                tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
                del output
                
                # Writing results
                for j in range(len(attention_prompts)):
                    generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
                    _util.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), layer, generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])

                del tokens, source_tokens, intervene_hooks
                torch.cuda.empty_cache()
                gc.collect()

else:
    header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'layer', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
    filepath = _util.create_csv_file(f"experiments/activation_intervention/output/{model_type}/{result_dir}", f"{prompt_type[1:]}_{intervention_loc}.csv", header, overwrite=False)

    batch_size = 24
    for i in tqdm(range(0, len(prompts), batch_size)):
        # Preparing prompts and labels
        batch_rows = prompts.iloc[i:i+batch_size]
        if pd.notna(batch_rows.iloc[0]['factual_output']) and batch_rows.iloc[0]['factual_output']:
            factual_labels_str = [str(sum) for sum in batch_rows['factual_output'].tolist()]
        else:
            factual_labels_str = [str(sum) for sum in batch_rows['base_sum'].tolist()]
        factual_labels = tokenizer(factual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
        factual_labels = factual_labels.squeeze(1)
        if pd.notna(batch_rows.iloc[0]['counterfactual_output']) and batch_rows.iloc[0]['counterfactual_output']:
            counterfactual_labels_str = [str(sum) for sum in batch_rows['counterfactual_output'].tolist()]
        else:
            counterfactual_labels_str = [str(sum) for sum in batch_rows['source_sum'].tolist()]
        counterfactual_labels = tokenizer(counterfactual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
        counterfactual_labels = counterfactual_labels.squeeze(1)

        # Preparing intervention hooks
        for layer in range(len(model.model.layers)):
            intervene_hooks = []
            tokens, source_tokens, hook = prepare_batch_multitoken_intervention(model, tokenizer, layer, tok_pos_list, batch_rows['base_prompt'].tolist(), batch_rows['source_prompt'].tolist(), module_format=f"model.layers.{layer}.self_attn", pre_hook=False)
            intervene_hooks.append(hook)
            input_length = tokens["input_ids"].shape[1]

            # Forward pass
            with torch.no_grad():
                output = batch_intervene(model, tokens["input_ids"], intervene_hooks, attention_mask=tokens["attention_mask"], pre_hook=False)
            pred_toks = output.logits[:,-1,:].argmax(dim=-1)
            prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
            factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
            counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
            tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
            del output
            
            # Writing results
            for j, (_, row) in enumerate(batch_rows.iterrows()):
                generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
                _util.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), layer, generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])

            del tokens, source_tokens, intervene_hooks
            torch.cuda.empty_cache()
            gc.collect()

  0%|                                                                                                                        | 0/11 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [46:27<00:00, 253.42s/it]


In [ ]:
if freeze_attention:
    batch_size = 1
else:
    batch_size = 24

header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'layer', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
filepath = _util.create_csv_file(f"experiments/activation_intervention/output/{model_type}/{result_dir}", f"{intervention_loc}_{prompt_type[1:]}.csv", header, overwrite=False)

for i in tqdm(range(0, len(prompts), batch_size)):
    # Preparing prompts and labels
    batch_rows = prompts.iloc[i:i+batch_size]
    if pd.notna(batch_rows.iloc[0]['factual_output']) and batch_rows.iloc[0]['factual_output']:
        factual_labels_str = [str(sum) for sum in batch_rows['factual_output'].tolist()]
    else:
        factual_labels_str = [str(sum) for sum in batch_rows['base_sum'].tolist()]
    factual_labels = tokenizer(factual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
    factual_labels = factual_labels.squeeze(1)
    if pd.notna(batch_rows.iloc[0]['counterfactual_output']) and batch_rows.iloc[0]['counterfactual_output']:
        counterfactual_labels_str = [str(sum) for sum in batch_rows['counterfactual_output'].tolist()]
    else:
        counterfactual_labels_str = [str(sum) for sum in batch_rows['source_sum'].tolist()]
    counterfactual_labels = tokenizer(counterfactual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"]
    counterfactual_labels = counterfactual_labels.squeeze(1)

    base_prompts = batch_rows['base_prompt'].tolist()
    source_prompts = batch_rows['source_prompt'].tolist()

    # If freeze_attention, repeat prompts and labels 20 times
    if freeze_attention:
        base_prompts = base_prompts * 20
        source_prompts = source_prompts * 20
        factual_labels = factual_labels.repeat(20)
        counterfactual_labels = counterfactual_labels.repeat(20)

    for layer in range(len(model.model.layers)):
        for head in range(64):
            # Preparing intervention hooks
            intervene_hooks = []
            tokens, source_tokens, hooks = prepare_batch_multitoken_head_intervention(model, tokenizer, [layer], tok_pos_list, base_prompts, source_prompts, head_idx=head)
            intervene_hooks += hooks
            input_length = tokens["input_ids"].shape[1]

            # Forward pass
            with torch.no_grad():
                output = batch_intervene(model, tokens["input_ids"], intervene_hooks+attention_freeze_hooks, attention_mask=tokens["attention_mask"])
            pred_toks = output.logits[:,-1,:].argmax(dim=-1)
            prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
            factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
            counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
            tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
            del output
            
            # Writing results
            if freeze_attention:
                for j in range(num_attention):
                    generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
                    _util.write_to_csv(filepath, batch_rows.iloc[0].to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])
            else:
                for j in range(batch_size):
                    generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
                    _util.write_to_csv(filepath, batch_rows.iloc[j].to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])

            del tokens, source_tokens, intervene_hooks
            torch.cuda.empty_cache()
            gc.collect()